<a href="https://colab.research.google.com/github/Xquisite-design/enterprise-presales-knowledge-orchestration/blob/main/notebooks/01_Data_Loading_Quality_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1 of 5 : Data Loading, Quality Assessment and Cleaning

Loads the Curated Enterprise Presales Research Corpus (CEPRC), assesses data quality and applies cleaning.
The cleaned dataset is saved to Drive for the remaining notebooks.

**Next:** `02_Exploratory_Data_Analysis.ipynb`

The Curated Enterprise Presales Research Corpus (CEPRC) consists of 900 annotated enterprise presales question–answer pairs represented by 29 variables. The dataset contains a combination of 23 categorical variables, 3 numerical variables, and 3 placeholder variables intended for future enhancement. Overall, the dataset is compact (approximately 204 KB) and suitable for exploratory analysis, statistical modelling, and machine learning

Since our dataset is an annotated enterprise presales corpus (around 900 observations with mostly categorical variables), I have elected to do EDA which is research-question driven rather than simply showing histograms and boxplots.

We will do the analysis in this order:

1. Dataset loading and inspection
2. Data quality assessment
3. Data cleaning log
4. EDA (Univariate and Bivariate EDA (aligned to each Research Question)
5. Inferential Statistics (RQ specific)
6. Feature engineering
7. Baseline ML models
8. Preliminary evaluation
9. GitHub documentation

#PART-A : EDA

## 1. Dataset loading and inspection

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df = pd.read_excel("/content/drive/MyDrive/DataAnalytics_Capstone/Enterprise_Presales_Knowledge_Research_Dataset_V1.xlsx", sheet_name='Enterprise_QA_Dataset')

print(df.shape)
print(df.info())

print(df.head())

print(df.describe(include='all'))

print(df.nunique())

print(df.isnull().sum())

print(df.duplicated().sum())

(900, 29)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 29 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Question_ID                       900 non-null    object 
 1   Vendor                            900 non-null    object 
 2   Source Website                    900 non-null    object 
 3   Question URL                      900 non-null    object 
 4   Source URL                        900 non-null    object 
 5   Question                          900 non-null    object 
 6   Official Answer                   900 non-null    object 
 7   Official Answer Summary           900 non-null    object 
 8   Official Documentation URL(s)     900 non-null    object 
 9   Knowledge Asset Category          900 non-null    object 
 10  Technology Domain                 900 non-null    object 
 11  Cloud Platform                    900 non-null    object 
 12

The above initial quality assessment indicates that the dataset is highly complete. All core research variables contain 900 non-null observations, while Publication Date, Accepted Answer Available?, and Accepted Answer URL contain no values because these fields were intentionally excluded from the current corpus. No duplicate observations were identified, confirming that each record represents a unique enterprise presales question–answer pair

Initial Observations

A few notable characteristics of the corpus emerge from the summary statistics:

- The dataset covers 8 technology vendors and 8 source websites, providing diversity across enterprise platforms.
- Question Type and Question Intent each contain 7 categories, aligning with the proposed inferential analyses for RQ2 and RQ4.
- The target variable Public Knowledge Only? contains two classes (Yes and Partially), confirming that Binary Logistic Regression is the appropriate statistical model for RQ1.
- Enterprise Knowledge Required contains 4 categories, supporting the planned Chi-square analyses.
- The variable Estimated RFP Relevance (1–5) has a mean score of 4.87, suggesting that the curated corpus primarily consists of questions that are highly relevant to enterprise presales.

## 2. Dataset Quality Assessment

Before beginning exploratory analysis, the dataset is examined for completeness, duplicate records, data types, missing values, and the number of unique categories in each variable. This assessment helps identify potential data quality issues that may affect subsequent statistical analysis and machine learning models.

In [3]:
###############################################################
# DATA QUALITY ASSESSMENT
#
# This section evaluates the overall quality of the dataset.
# The following checks are performed:
#
# 1. Dataset dimensions
# 2. Data types
# 3. Missing values
# 4. Duplicate records
# 5. Unique values per variable
###############################################################

import pandas as pd

# Display number of rows and columns
print("="*60)
print("DATASET DIMENSIONS")
print("="*60)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

# Display data types
print("\n")
print("="*60)
print("COLUMN DATA TYPES")
print("="*60)
print(df.dtypes)

# Missing values
print("\n")
print("="*60)
print("MISSING VALUES")
print("="*60)

missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': round(df.isnull().mean()*100,2)
})

print(missing)

# Duplicate records
print("\n")
print("="*60)
print("DUPLICATE RECORDS")
print("="*60)

duplicates = df.duplicated().sum()

print(f"Duplicate Rows : {duplicates}")

# Number of unique values
print("\n")
print("="*60)
print("UNIQUE VALUES PER COLUMN")
print("="*60)

unique_values = pd.DataFrame({
    'Unique Values': df.nunique()
})

print(unique_values)

DATASET DIMENSIONS
Rows    : 900
Columns : 29


COLUMN DATA TYPES
Question_ID                          object
Vendor                               object
Source Website                       object
Question URL                         object
Source URL                           object
Question                             object
Official Answer                      object
Official Answer Summary              object
Official Documentation URL(s)        object
Knowledge Asset Category             object
Technology Domain                    object
Cloud Platform                       object
Presales Category                    object
Business Capability                  object
Question Intent                      object
Question Type                        object
Question Complexity                  object
Question Length                       int64
Number of Technologies Mentioned      int64
Document Type                        object
Publication Date                    float64
Tags      

The dataset was manually curated and annotated prior to analysis. Consequently, only basic quality checks such as missing values, duplicate records, and consistency of categorical labels are performed. Any identified issues are documented before proceeding with the exploratory analysis.

## 3. Data Cleaning Log

In [4]:
###############################################################
# DATA CLEANING
#
# The purpose of this section is to identify any issues that
# require cleaning before statistical analysis.
#
# Checks performed:
#
# - Missing values
# - Duplicate records
# - Extra white spaces
# - Inconsistent category labels
###############################################################

# Remove leading and trailing spaces from all text columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Check missing values again
cleaning_log = pd.DataFrame({
    "Variable": df.columns,
    "Missing": df.isnull().sum(),
    "Unique Categories": df.nunique()
})

display(cleaning_log)

print("\nData Cleaning Completed.")

,Variable,Missing,Unique Categories
Question_ID,Question_ID,0,900
Vendor,Vendor,0,8
Source Website,Source Website,0,8
Question URL,Question URL,0,98
Source URL,Source URL,0,75
Question,Question,0,896
Official Answer,Official Answer,0,900
Official Answer Summary,Official Answer Summary,0,899
Official Documentation URL(s),Official Documentation URL(s),0,74
Knowledge Asset Category,Knowledge Asset Category,0,10



Data Cleaning Completed.


## Save Cleaned Dataset

Save the cleaned dataset so the following notebooks can load it without repeating the steps above.

In [ ]:
###############################################################
# SAVE CLEANED DATASET FOR THE NEXT NOTEBOOKS
###############################################################

import os

BASE_DIR = "/content/drive/MyDrive/DataAnalytics_Capstone"
HANDOFF_DIR = f"{BASE_DIR}/handoff"

os.makedirs(HANDOFF_DIR, exist_ok=True)

df.to_pickle(f"{HANDOFF_DIR}/cleaned_dataset.pkl")

print("Cleaned dataset saved.")